# Split Conformal — California Housing (MLP)

In [ ]:
import numpy as np
import torch, torch.nn as nn, json, os, sys
from sklearn.metrics import r2_score
from sklearn.neighbors import KNeighborsRegressor
sys.path.insert(0, os.path.join('.', '..'))
from shared.data_utils import load_california_housing, build_mlp_encoder

CONFIG = {'method': 'conformal', 'hidden_dims': [128, 64], 'cal_ratio': 0.3, 'k_neighbors': 20,
          'epochs': 200, 'batch_size': 32, 'lr': 1e-3, 'seeds': [42, 43, 44]}
RESULT_DIR = os.path.join('.', 'results', 'conformal')
os.makedirs(RESULT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
class BaseRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dims):
        super().__init__()
        self.encoder, h_dim = build_mlp_encoder(input_dim, hidden_dims)
        self.head = nn.Linear(h_dim, 1)
    def forward(self, x): return self.head(self.encoder(x))
print('Model defined.')

In [ ]:
def train_one_seed(seed):
    print(f'\n--- Seed {seed} ---')
    X_train, y_train, X_val, y_val, X_test, y_test, scaler, input_dim = \
        load_california_housing(random_state=seed)
    # 从训练集中划出校准集
    n_cal = int(len(X_train) * CONFIG['cal_ratio'])
    X_cal, y_cal = X_train[-n_cal:], y_train[-n_cal:]
    X_tr, y_tr = X_train[:-n_cal], y_train[:-n_cal]
    train_ds = torch.utils.data.TensorDataset(X_tr, y_tr)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
    model = BaseRegressor(input_dim, CONFIG['hidden_dims']).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
    crit = nn.MSELoss()
    best_state, best_val = None, float('inf')
    for _ in range(CONFIG['epochs']):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad(): vl = crit(model(X_val.to(DEVICE)), y_val.to(DEVICE)).item()
        if vl < best_val: best_val = vl; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    # kNN 局部残差预测
    with torch.no_grad():
        cal_feat = model.encoder(X_cal.to(DEVICE)).cpu().numpy()
        cal_pred = model.head(torch.tensor(cal_feat, dtype=torch.float32, device=DEVICE)).cpu().numpy().squeeze()
        test_feat = model.encoder(X_test.to(DEVICE)).cpu().numpy()
        y_pred = model.head(torch.tensor(test_feat, dtype=torch.float32, device=DEVICE)).cpu().numpy().squeeze()
    cal_resid = np.abs(y_cal.numpy().squeeze() - cal_pred)
    knn = KNeighborsRegressor(n_neighbors=min(CONFIG['k_neighbors'], len(cal_feat)))
    knn.fit(cal_feat, cal_resid)
    scores = -knn.predict(test_feat)  # 负残差：越高越好
    return y_pred, scores, y_test.numpy().squeeze()

for seed in CONFIG['seeds']:
    y_pred, scores, y_true = train_one_seed(seed)
    sd = os.path.join(RESULT_DIR, f'seed_{seed}'); os.makedirs(sd, exist_ok=True)
    np.save(os.path.join(sd, 'test_predictions.npy'), y_pred)
    np.save(os.path.join(sd, 'test_scores.npy'), scores)
    np.save(os.path.join(sd, 'test_labels.npy'), y_true)
    print(f'  R²: {r2_score(y_true, y_pred):.4f}')
print(f'\nDone. Saved to {RESULT_DIR}')